In [ ]:
# R
# Ensure required packages are installed and loaded
# Ejemplo completo de ANOVA con detección automática de columnas
library(readr)
library(dplyr)
library(tidyr)
library(purrr)
library(ggplot2)
library(skimr)

# 1. Leer datos
df <- readr::read_csv("../../data/breast_cancer_data.csv", show_col_types = FALSE)
df

# A tibble: 569 × 31
   `mean radius` `mean texture` `mean perimeter` `mean area` `mean smoothness` `mean compactness`
           <dbl>          <dbl>            <dbl>       <dbl>             <dbl>              <dbl>
 1          18.0           10.4            123.        1001             0.118              0.278 
 2          20.6           17.8            133.        1326             0.0847             0.0786
 3          19.7           21.2            130         1203             0.110              0.160 
 4          11.4           20.4             77.6        386.            0.142              0.284 
 5          20.3           14.3            135.        1297             0.100              0.133 
 6          12.4           15.7             82.6        477.            0.128              0.17  
 7          18.2           20.0            120.        1040             0.0946             0.109 
 8          13.7           20.8             90.2        578.            0.119              0.164 

In [7]:
# 2. EDA Completo sobre df

## 2.1 Dimensiones y tipos de datos
glimpse(df)

## 2.2 Resumen estadístico
skimr::skim(df)

## 2.3 Valores faltantes por columna
missing_summary <- df |>
  summarise(across(everything(), ~ sum(is.na(.)))) |>
  pivot_longer(everything(), names_to = "variable", values_to = "n_missing") |>
  filter(n_missing > 0) |>
  arrange(desc(n_missing))
missing_summary

## 2.4 Distribución de la variable target
df |>
  count(target) |>
  mutate(pct = n / sum(n) * 100)

## 2.5 Distribución de variables numéricas
num_vars <- df |> select(where(is.numeric), -target) |> names()

df |>
  select(all_of(num_vars)) |>
  pivot_longer(everything(), names_to = "variable", values_to = "value") |>
  ggplot(aes(x = value)) +
  geom_histogram(bins = 30, fill = "steelblue", color = "white", alpha = 0.8) +
  facet_wrap(~variable, scales = "free") +
  theme_minimal() +
  labs(title = "Distribución de variables numéricas")

## 2.6 Boxplots por target
df |>
  select(target, all_of(num_vars)) |>
  mutate(target = factor(target)) |>
  pivot_longer(-target, names_to = "variable", values_to = "value") |>
  ggplot(aes(x = target, y = value, fill = target)) +
  geom_boxplot(alpha = 0.7, outlier.alpha = 0.3) +
  facet_wrap(~variable, scales = "free_y") +
  theme_minimal() +
  labs(title = "Distribución por clase (target)") +
  theme(legend.position = "none")

## 2.7 Matriz de correlación
cor_matrix <- df |>
  select(all_of(num_vars)) |>
  cor(use = "pairwise.complete.obs")

cor_matrix |>
  as.data.frame() |>
  tibble::rownames_to_column("var1") |>
  pivot_longer(-var1, names_to = "var2", values_to = "correlation") |>
  ggplot(aes(x = var1, y = var2, fill = correlation)) +
  geom_tile() +
  scale_fill_gradient2(low = "blue", mid = "white", high = "red", midpoint = 0) +
  theme_minimal() +
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  labs(title = "Matriz de correlación")

## 2.8 Variables más correlacionadas con target
df |>
  mutate(target = as.numeric(target)) |>
  select(all_of(num_vars), target) |>
  cor(use = "pairwise.complete.obs") |>
  as.data.frame() |>
  tibble::rownames_to_column("variable") |>
  select(variable, cor_with_target = target) |>
  filter(variable != "target") |>
  arrange(desc(abs(cor_with_target)))

Rows: 569
Columns: 31
$ `mean radius`             <dbl> 17.990, 20.570, 19.690, 11.420, 20.290, 12.450, 18.250, 13.710, 13.000…
$ `mean texture`            <dbl> 10.38, 17.77, 21.25, 20.38, 14.34, 15.70, 19.98, 20.83, 21.82, 24.04, …
$ `mean perimeter`          <dbl> 122.80, 132.90, 130.00, 77.58, 135.10, 82.57, 119.60, 90.20, 87.50, 83…
$ `mean area`               <dbl> 1001.0, 1326.0, 1203.0, 386.1, 1297.0, 477.1, 1040.0, 577.9, 519.8, 47…
$ `mean smoothness`         <dbl> 0.11840, 0.08474, 0.10960, 0.14250, 0.10030, 0.12780, 0.09463, 0.11890…
$ `mean compactness`        <dbl> 0.27760, 0.07864, 0.15990, 0.28390, 0.13280, 0.17000, 0.10900, 0.16450…
$ `mean concavity`          <dbl> 0.30010, 0.08690, 0.19740, 0.24140, 0.19800, 0.15780, 0.11270, 0.09366…
$ `mean concave points`     <dbl> 0.14710, 0.07017, 0.12790, 0.10520, 0.10430, 0.08089, 0.07400, 0.05985…
$ `mean symmetry`           <dbl> 0.2419, 0.1812, 0.2069, 0.2597, 0.1809, 0.2087, 0.1794, 0.2196, 0.2350…
$ `mean fractal dimensio

: [1m[33mError[39m in `loadNamespace()`:[22m
[33m![39m there is no package called ‘skimr’